In [ ]:
# 1 - Download, preprocess and split the dataset if necessary. Then create a dataloader for each split

from datasets import load_dataset
import torch
from torch.utils.data import Subset
import random
from dataset import ImageDataset
from degradation import ImageDegradation, DegradationParameters, RGBBlurOperator
from utils import plot # type: ignore
from config import TRAIN_SIZE, VALIDATION_SIZE, TEST_SIZE
from IPPy import operators
from models import NAFNet, NAFNetModel, HQSNet, HQSNetModel

# Load ImageNet 256x256 from HuggingFace
ds = load_dataset(
    "benjamin-paine/imagenet-1k-256x256"
)

# Apply preprocessing through the custom Dataset class
# Apply preprocessing through the custom Dataset class
train_dataset = Subset(
    ImageDataset(ds["train"]),
    range(TRAIN_SIZE),
)

validation_dataset = Subset(
    ImageDataset(ds["validation"]),
    range(VALIDATION_SIZE),
)

test_dataset = Subset(
    ImageDataset(ds["test"]),
    range(TEST_SIZE),
)

train_degradation_restoration = ImageDegradation(
    DegradationParameters(
        image_size=256,
        kernel_type="motion",
        kernel_size=9,
        motion_angle=45.0,
    )
)

K_rgb = RGBBlurOperator(train_degradation_restoration.operator)

naf_net = NAFNet(
        image_shape=(3,256,256),
        base_channels=32,
        enc_blocks=[1,2,3,4],
        dec_blocks=[2,2,2,1],
        middle_blocks=6
    )
naf_model = NAFNetModel(naf_net)

hqs_net = HQSNet(
    K=K_rgb,
    denoiser=naf_net,
    checkpoint_path="./weights/NAFNet/NAFImgDenoise.pth",
    n_layers=3,
    cg_iters=3,
    initial_mu=1.0,
)

hqs_model = HQSNetModel(hqs_net)

train_degradation_denoiser = ImageDegradation(
    DegradationParameters(
        image_size=256,
        kernel_type=None,
        noise_levels=[0.005, 0.01, 0.05, 0.1]
    )
)

validation_degradations_denoiser = [
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.005])),
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.01])),
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.05])),
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.1])),
]

validation_degradations_restoration = [
    ImageDegradation(DegradationParameters(noise_levels=[0.005])),
    ImageDegradation(DegradationParameters(noise_levels=[0.01])),
    ImageDegradation(DegradationParameters(noise_levels=[0.05])),
    ImageDegradation(DegradationParameters(noise_levels=[0.1])),
]

TRAIN_MODEL = True
if TRAIN_MODEL:
    naf_history = naf_model.train_model(
        n_epochs=50,
        train_dataset=train_dataset,
        validation_dataset=validation_dataset,
        train_degradation=train_degradation_denoiser,
        validation_degradations=validation_degradations_denoiser,
        batch_size=16,
        learning_rate=1e-4,
        checkpoint_path="./weights/HQSUnrolled/NAF_checkpoint.pth",
        resume=True,
        preview_every=5,
        preview_n=1
    )
    hsq_history = hqs_model.train_model(
        n_epochs=50,
        train_dataset=train_dataset,
        validation_dataset=validation_dataset,
        train_degradation=train_degradation_restoration,
        validation_degradations=validation_degradations_restoration,
        batch_size=16,
        learning_rate=1e-3,
        checkpoint_path="./weights/HQSUnrolled/HQS_checkpoint.pth",
        resume=True,
        preview_every=5,
        preview_n=1
    )



Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

Resumed NAFNet training from epoch 50


HQS epoch 1/50:   0%|          | 0/625 [00:00<?, ?it/s]

KeyboardInterrupt: 